In [ ]:
# Цель этого урока — сгенерировать и сохранить ответы вашей RAG-системы 
# на все вопросы из «эталонного» набора данных (ground truth). Это 
# промежуточный, но критически важный этап для последующей оценки 
# качества всей RAG-системы.

# Проще говоря, вы не просто так запускаете RAG для каждого вопроса. 
# Вы создаете файл с ответами, который станет «сырьем» для следующего
# урока, где вы будете использовать LLM как судью (LLM-as-a-judge),
# чтобы оценить, насколько эти ответы хороши

In [ ]:
# Контекст

# A (answer) - ответ в разделе часто задаваемых вопросов
# Q (question) - вопрос генерируемый на основе ответа LLM
# A' (answer) - ответ, полученный нашей системой RAG при задании Q*.

# Мы сравниваем A' с A, чтобы проверить, выдала ли система 
# правильный ответ. 

# Основная идея в том, что если ответ вашей RAG-системы (A') 
# очень близок к оригинальному ответу (A), значит, система 
# работает хорошо. Поскольку вы знаете, из какого документа 
# (A) был сгенерирован каждый вопрос (Q), вы можете позже сравнить
# ответ RAG (A') с этим оригиналом (A). Это называется офлайн-оценкой, 
# потому что вы сравниваете с заранее известным «правильным» ответом, 
# а не полагаетесь на мнение пользователей в реальном времени.

In [1]:
# Загружаем вопросы для проверки RAG правильных вопросов

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
# Загрузите документы с часто задаваемыми вопросами и поисковый индекс:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

for doc in documents:
  if doc["course"] == "llm-zoomcamp":
    documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
# Создаем справочную таблицу для исходных документов с 
# часто задаваемыми вопросами:

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

doc_idx

{'74eb249bbf': {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 '977bf7786c': {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 '489dd1c9d9': {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
# RAGWithUsage - это вспомогательные функции оценки. После каждого 
# вызова LLM-функции система сохраняет данные об использовании 
# токенов. Затем мы можем рассчитать общую стоимость.

In [7]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
  index = index,
  llm_client = openai_client,
)

In [ ]:
# Для каждого вопроса RAGBaseсистема выполняет поиск в 
# разделе часто задаваемых вопросов (FAQ), формирует 
# подсказку на основе полученного контекста и просит 
# LLM ответить.

In [ ]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join and follow along. You can start whenever you want, and the videos/materials are available.\n\nIf you want a certificate, though, you need to submit your project while a live cohort is still accepting submissions, and complete the required peer reviews.'

In [ ]:
# Проверяем сколько этот вызов нам обошелся - 0.0007365
assistant.total_cost()

0.0007365

In [ ]:
# Получаем исходный ответ из документа с идентификатором:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [ ]:
# Теперь сохраняем оба результата в одну запись
rag_result = {
  "question": question,
  "answer_llm": answer_llm,
  "answer_orig": answer_orig,
  "document": doc_id,
}

rag_result

{'question': 'I found this course late — can I still enroll and follow along?',
 'answer_llm': 'Yes, you can still join and follow along. You can start whenever you want, and the videos/materials are available.\n\nIf you want a certificate, though, you need to submit your project while a live cohort is still accepting submissions, and complete the required peer reviews.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [ ]:
# Сбрасываем данные об использовании
assistant.reset_usage()

In [ ]:
# Теперь оборачиваем все в одну функцию, которая выполняет
# все действия предыдущего запроса

def generate_rag_answer(rec):
  # получаем вопрос, вопрос, id документа
  question = rec["question"]
  doc_id = rec["document"]
  original_doc = doc_idx[doc_id]

  # ищем его и отправляеи через rag
  answer_llm = assistant.rag(question)
  answer_orig = original_doc["answer"]

  result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
  }

  return result

answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I found this course late — can I still enroll and follow along?',
 'answer_llm': 'Yes — you can still join and follow along. The course materials are available, and you can start whenever you want.\n\nIf you want a certificate, though, you need to:\n- finish with a live cohort,\n- submit your capstone project while submissions are still being accepted,\n- and complete the required peer reviews.\n\nYou can still work through the content in self-paced mode, but the certificate requires the live cohort submission/review window.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers = 6) as pool:
  results = map_progress(pool, ground_truth, generate_rag_answer)

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [9]:
assistant.total_cost()

0.0

In [ ]:
df_answers = pd.DataFrame(answers)
# df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [ ]:
# Что конкретно делает в коде 

# С практической точки зрения, урок проводит вас через следующие шаги:
# - Загрузка данных: Вы загружаете «эталонные» вопросы (ground truth) 
# и документы FAQ.
# - Настройка RAG: Вы инициализируете RAG-систему (класс RAGWithUsage), 
# которая умеет искать по индексу и генерировать ответы, а также 
# подсчитывать затраты на токены.
# - Генерация ответов: Вы запускаете RAG для каждого вопроса. Система 
# ищет контекст, формирует промпт и просит LLM ответить.
# - Сохранение результатов: Для каждого вопроса вы сохраняете в одну 
# запись: сам вопрос, ответ RAG (answer_llm) и оригинальный ответ
# (answer_orig).
# - Пакетная обработка: Вы обрабатываете все вопросы из набора (в примере
# — 395 штук), используя параллельные потоки для ускорения, и сохраняете
#  все результаты в итоговый CSV-файл (data/rag-answers-new.csv).

# Этот файл и есть главный артефакт урока. Он — мост к следующему 
# занятию, где вы будете оценивать качество этих ответов.